In [ ]:
import shutil
from pathlib import Path
import kagglehub
owner = "sabahesaraki"
dataset = "breast-ultrasound-images-dataset"
cache_path = kagglehub.dataset_download(f"{owner}/{dataset}")

target_dir = Path("../../data")
target_dir.mkdir(parents=True, exist_ok=True)
print("복사 중 →", target_dir.resolve())
shutil.copytree(cache_path, target_dir, dirs_exist_ok=True)
print("완료! 최종 위치:", target_dir.resolve())

In [ ]:
import os, random, time, csv
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# --- 경로/설정 ---
DATA_ROOT = "../../data/Dataset_BUSI_with_GT"
OUTPUT    = "./outputs_busi_cls"

IMG_SIZE    = 384
BATCH_SIZE  = 16
EPOCHS      = 50
LR          = 3e-4
WEIGHT_DECAY= 1e-4
SPLIT_RATIO = (0.7, 0.1, 0.2)  # train/val/test
SEED        = 42
AMP         = True

os.makedirs(OUTPUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
device


In [ ]:
LABEL_TO_IDX = {"normal": 0, "benign": 1, "malignant": 2}
IDX_TO_LABEL = {v:k for k,v in LABEL_TO_IDX.items()}

def build_index_from_folders(root):
    root = Path(root)
    items = []
    for cls in ("normal", "benign", "malignant"):
        d = root/cls
        if not d.exists():
            print(f"[경고] 폴더 없음: {d}")
            continue
        for ext in ("*.png","*.jpg","*.jpeg","*.bmp","*.tif"):
            for p in sorted(d.rglob(ext)):
                # 🔴 분류에서는 마스크 파일 제외
                if "_mask" in p.name.lower():
                    continue
                items.append((str(p), LABEL_TO_IDX[cls]))
    return items

items = build_index_from_folders(DATA_ROOT)
print("총 이미지 수:", len(items),
      "| 라벨 분포:", {k: sum(1 for _,y in items if y==v) for k,v in LABEL_TO_IDX.items()})

# 필요하면 CSV로 저장(옵션)
csv_path = os.path.join(OUTPUT, "busi_cls_index.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["path","label"])  # label은 0/1/2
    for p,l in items: w.writerow([p,l])
csv_path


In [ ]:
def stratified_split(items, split=(0.7,0.1,0.2), seed=42):
    rng = random.Random(seed)
    by_label = {}
    for it in items:
        by_label.setdefault(it[1], []).append(it)
    for k in by_label: rng.shuffle(by_label[k])
    train, val, test = [], [], []
    for k, lst in by_label.items():
        n = len(lst)
        n_train = int(n*split[0])
        n_val   = int(n*split[1])
        train += lst[:n_train]
        val   += lst[n_train:n_train+n_val]
        test  += lst[n_train+n_val:]
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

train_items, val_items, test_items = stratified_split(items, split=SPLIT_RATIO, seed=SEED)
len(train_items), len(val_items), len(test_items)


In [ ]:
class BUSIClsDataset(Dataset):
    def __init__(self, items, img_size=384, is_train=False):
        self.items = items
        self.is_train = is_train
        self.size = img_size
        self.tf_train = transforms.Compose([
            transforms.Resize((self.size, self.size)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(0.1, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        self.tf_eval = transforms.Compose([
            transforms.Resize((self.size, self.size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert("RGB")
        img = self.tf_train(img) if self.is_train else self.tf_eval(img)
        return img, label, path

train_ds = BUSIClsDataset(train_items, IMG_SIZE, True)
val_ds   = BUSIClsDataset(val_items,   IMG_SIZE, False)
test_ds  = BUSIClsDataset(test_items,  IMG_SIZE, False)

# cuda
# train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
# val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
# test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
# mac
train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

next(iter(train_ld))[0].shape  # (B, 3, H, W)


In [ ]:
def build_model(num_classes=3):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

model = build_model(3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=AMP)

model


In [ ]:
def accuracy_from_logits(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

def macro_f1_from_logits(logits, y, num_classes=3):
    with torch.no_grad():
        yp = logits.argmax(1)
        cm = torch.zeros((num_classes,num_classes), dtype=torch.long, device=logits.device)
        for t,p in zip(y.view(-1), yp.view(-1)):
            cm[t,p] += 1
        f1s=[]
        for c in range(num_classes):
            tp = cm[c,c].float()
            fp = cm[:,c].sum().float()-tp
            fn = cm[c,:].sum().float()-tp
            denom = (2*tp+fp+fn)
            f1 = (2*tp/denom) if denom>0 else torch.tensor(0.0, device=logits.device)
            f1s.append(f1)
        return torch.stack(f1s).mean().item()

def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss=total_acc=total_f1=0.0
    n=0
    for x,y,_ in loader:
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.cuda.amp.autocast(enabled=AMP):
            logits = model(x)
            loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        bs=x.size(0)
        total_loss += loss.item()*bs
        total_acc  += accuracy_from_logits(logits,y)*bs
        total_f1   += macro_f1_from_logits(logits,y)*bs
        n += bs
    return total_loss/n, total_acc/n, total_f1/n


In [ ]:
best_val_f1, best_ckpt = -1, None
history = {"tr_loss":[], "tr_acc":[], "tr_f1":[], "va_loss":[], "va_acc":[], "va_f1":[]}

for epoch in range(1, EPOCHS+1):
    t0=time.time()
    tr_loss,tr_acc,tr_f1 = run_epoch(model, train_ld, True)
    va_loss,va_acc,va_f1 = run_epoch(model, val_ld,   False)
    scheduler.step()

    history["tr_loss"].append(tr_loss); history["tr_acc"].append(tr_acc); history["tr_f1"].append(tr_f1)
    history["va_loss"].append(va_loss); history["va_acc"].append(va_acc); history["va_f1"].append(va_f1)

    if va_f1>best_val_f1:
        best_val_f1=va_f1
        best_ckpt = os.path.join(OUTPUT, "best.pt")
        torch.save({"model":model.state_dict(),"epoch":epoch,"val_f1":va_f1}, best_ckpt)

    print(f"[{epoch:03d}/{EPOCHS}] "
          f"train L{tr_loss:.4f} A{tr_acc:.4f} F1{tr_f1:.4f} | "
          f"val L{va_loss:.4f} A{va_acc:.4f} F1{va_f1:.4f} | "
          f"{time.time()-t0:.1f}s")

print("Best val macro-F1:", best_val_f1, "| ckpt:", best_ckpt)


In [ ]:
if best_ckpt and os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded epoch {ckpt['epoch']}  val_f1={ckpt['val_f1']:.4f}")

model.eval()
all_logits=[]; all_labels=[]
with torch.no_grad():
    for x,y,_ in test_ld:
        x=x.to(device); y=y.to(device)
        logits = model(x)
        all_logits.append(logits.cpu()); all_labels.append(y.cpu())
all_logits = torch.cat(all_logits); all_labels = torch.cat(all_labels)

test_acc = accuracy_from_logits(all_logits, all_labels)
test_f1  = macro_f1_from_logits(all_logits, all_labels)
print(f"[TEST] Acc={test_acc:.4f}  Macro-F1={test_f1:.4f}")

y_pred = all_logits.argmax(1).numpy()
y_true = all_labels.numpy()
cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
disp = ConfusionMatrixDisplay(cm, display_labels=["normal","benign","malignant"])
fig, ax = plt.subplots(figsize=(4.5,4))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.show()

print(classification_report(y_true, y_pred, target_names=["normal","benign","malignant"], digits=4))


In [ ]:
model.eval()
imgs, labels, paths = next(iter(test_ld))
with torch.no_grad():
    logits = model(imgs.to(device))
preds = logits.argmax(1).cpu().numpy()
labels = labels.numpy()

def show_grid(imgs, labels, preds, n=8):
    n = min(n, len(imgs))
    plt.figure(figsize=(14,6))
    for i in range(n):
        plt.subplot(2, (n+1)//2, i+1)
        img = imgs[i].permute(1,2,0).numpy()
        img = (img * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406]))
        img = np.clip(img, 0, 1)
        plt.imshow(img)
        title = f"T:{IDX_TO_LABEL[int(labels[i])]}  P:{IDX_TO_LABEL[int(preds[i])]}"
        plt.title(title, color=("green" if labels[i]==preds[i] else "red"))
        plt.axis("off")
    plt.tight_layout(); plt.show()

show_grid(imgs, labels, preds, n=8)


In [ ]:
import os, random, time, csv
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF

import matplotlib.pyplot as plt

# --- 경로/설정 ---
DATA_ROOT = "./data/BUSI/Dataset_BUSI_with_GT"   # 예: ".../BUSI" (normal/benign/malignant 하위 폴더)
OUTPUT    = "./outputs_busi_seg"

IMG_SIZE    = 384
BATCH_SIZE  = 8
EPOCHS      = 60
LR          = 3e-4
WEIGHT_DECAY= 1e-4
SPLIT_RATIO = (0.7, 0.1, 0.2)  # train/val/test
SEED        = 42
AMP         = True

os.makedirs(OUTPUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
device


In [ ]:
def list_pairs_from_busi(root):
    root = Path(root)
    pairs = []  # (img_path, mask_path, class_idx)  # class_idx: 0=benign, 1=malignant
    for cls, cidx in [("benign",0), ("malignant",1)]:
        d = root/cls
        if not d.exists():
            print(f"[경고] 폴더 없음: {d}")
            continue
        imgs = []
        masks = {}
        # 마스크 먼저 인덱스
        for p in sorted(d.rglob("*_mask.*")):
            base = p.name.replace("_mask","")
            masks[base] = str(p)
        # 이미지 매칭
        for p in sorted(d.rglob("*.*")):
            name = p.name
            if name.endswith((".png",".jpg",".jpeg",".bmp",".tif")) and ("_mask" not in name):
                if name in masks:
                    pairs.append((str(p), masks[name], cidx))
    return pairs

pairs = list_pairs_from_busi(DATA_ROOT)
print("총 페어 수:", len(pairs))
print("benign:", sum(1 for _,_,c in pairs if c==0), " | malignant:", sum(1 for _,_,c in pairs if c==1))


In [ ]:
def stratified_split_pairs(pairs, split=(0.7,0.1,0.2), seed=42):
    by_label = {}
    for it in pairs:
        by_label.setdefault(it[2], []).append(it)
    rng = random.Random(seed)
    for k in by_label: rng.shuffle(by_label[k])
    train, val, test = [], [], []
    for k, lst in by_label.items():
        n = len(lst)
        n_train = int(n*split[0]); n_val = int(n*split[1])
        train += lst[:n_train]
        val   += lst[n_train:n_train+n_val]
        test  += lst[n_train+n_val:]
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

train_items, val_items, test_items = stratified_split_pairs(pairs, split=SPLIT_RATIO, seed=SEED)
len(train_items), len(val_items), len(test_items)


In [ ]:
class BUSISegDataset(Dataset):
    def __init__(self, items, img_size=384, is_train=False, return_cls=False):
        self.items = items
        self.size = img_size
        self.is_train = is_train
        self.return_cls = return_cls  # 필요 시 양성/악성 클래스 제공

    def __len__(self): return len(self.items)

    def _augment(self, img, mask):
        # 랜덤 플립
        if random.random() < 0.5:
            img = TF.hflip(img); mask = TF.hflip(mask)
        # 랜덤 회전(±10도)
        angle = random.uniform(-10, 10)
        img  = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST)
        # 밝기/대비 약하게
        if random.random() < 0.5:
            img = TF.adjust_brightness(img, 1.0 + random.uniform(-0.1, 0.1))
            img = TF.adjust_contrast(img,  1.0 + random.uniform(-0.1, 0.1))
        return img, mask

    def __getitem__(self, idx):
        img_path, mask_path, cidx = self.items[idx]
        img  = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # 단일 채널

        img  = TF.resize(img,  (self.size, self.size), interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, (self.size, self.size), interpolation=TF.InterpolationMode.NEAREST)

        if self.is_train:
            img, mask = self._augment(img, mask)

        img  = TF.to_tensor(img)
        img  = TF.normalize(img, [0.485,0.456,0.406],[0.229,0.224,0.225])

        mask = TF.to_tensor(mask)           # [1,H,W], 0~1
        mask = (mask > 0.5).float()         # 이진화

        if self.return_cls:
            return img, mask, cidx, img_path
        else:
            return img, mask, img_path

train_ds = BUSISegDataset(train_items, IMG_SIZE, is_train=True,  return_cls=False)
val_ds   = BUSISegDataset(val_items,   IMG_SIZE, is_train=False, return_cls=False)
test_ds  = BUSISegDataset(test_items,  IMG_SIZE, is_train=False, return_cls=False)

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

imgs, masks, paths = next(iter(train_ld))
imgs.shape, masks.shape, paths[0]


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=3):
        super().__init__()
        p = k // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    """
    업샘플 후 skip과 concat → DoubleConv
    ch_in: 업샘플된 feature의 채널 수
    ch_skip: 스킵(feature)의 채널 수
    out_ch: 출력 채널 수
    """
    def __init__(self, ch_in, ch_skip, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = DoubleConv(ch_in + ch_skip, out_ch)  # ★ concat 후 채널 합을 in_ch로 사용

    def forward(self, x, skip):
        x = self.up(x)
        # 혹시 1px 차이 생기면 정렬
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        if dy != 0 or dx != 0:
            x = TF.pad(x, [0, 0, max(dx, 0), max(dy, 0)])
            x = x[:, :, :skip.size(2), :skip.size(3)]
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, base=64, num_classes=1):
        super().__init__()
        self.inc   = DoubleConv(in_ch, base)         # C = b
        self.down1 = Down(base, base*2)              # 2b
        self.down2 = Down(base*2, base*4)            # 4b
        self.down3 = Down(base*4, base*8)            # 8b
        self.down4 = Down(base*8, base*16)           # 16b (bottleneck)

        # Up(ch_in, ch_skip, out_ch)
        self.up1  = Up(base*16, base*8,  base*8)     # (16b up) + 8b -> 8b
        self.up2  = Up(base*8,  base*4,  base*4)     # (8b  up) + 4b -> 4b
        self.up3  = Up(base*4,  base*2,  base*2)     # (4b  up) + 2b -> 2b
        self.up4  = Up(base*2,  base,    base)       # (2b  up) +  b ->  b

        self.outc = nn.Conv2d(base, num_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)   # b
        x2 = self.down1(x1)# 2b
        x3 = self.down2(x2)# 4b
        x4 = self.down3(x3)# 8b
        x5 = self.down4(x4)#16b

        u1 = self.up1(x5, x4)  # 8b
        u2 = self.up2(u1, x3)  # 4b
        u3 = self.up3(u2, x2)  # 2b
        u4 = self.up4(u3, x1)  #  b

        logits = self.outc(u4) # num_classes
        return logits

# 간단한 shape 체크 (옵션)
with torch.no_grad():
    _x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    _m = UNet(in_ch=3, base=64, num_classes=1).to(device)
    _y = _m(_x)
    print("out shape:", _y.shape)  # [2, 1, H, W]


In [ ]:
def dice_coeff(pred, target, eps=1e-6):
    # pred: 로짓 또는 확률; 여기선 시그모이드 확률로 변환 후 사용
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (union + eps)
    return dice.mean().item()

def iou_score(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    iou = (inter + eps) / (union + eps)
    return iou.mean().item()

class BCEDiceLoss(nn.Module):
    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dw = dice_weight
    def forward(self, logits, target):
        bce = self.bce(logits, target)
        # Soft Dice (확률 기반)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(1,2,3))
        union = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
        dice = 1 - (2*inter + 1e-6) / (union + 1e-6)
        return (1-self.dw)*bce + self.dw*dice.mean()

model = UNet(in_ch=3, base=64, num_classes=1).to(device)
criterion = BCEDiceLoss(dice_weight=0.5)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=AMP)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss=0.0; total_dice=0.0; total_iou=0.0; n=0
    for imgs, masks, _ in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks= masks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=AMP):
            logits = model(imgs)
            loss   = criterion(logits, masks)

        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()

        bs = imgs.size(0)
        total_loss += loss.item()*bs
        total_dice += dice_coeff(logits.detach(), masks)*bs
        total_iou  += iou_score(logits.detach(), masks)*bs
        n += bs
    return total_loss/n, total_dice/n, total_iou/n


In [ ]:
best_val_dice, best_ckpt = -1, None
history = {"tr_loss":[], "tr_dice":[], "tr_iou":[], "va_loss":[], "va_dice":[], "va_iou":[]}

for epoch in range(1, EPOCHS+1):
    t0=time.time()
    tr_loss,tr_dice,tr_iou = run_epoch(train_ld, train=True)
    va_loss,va_dice,va_iou = run_epoch(val_ld,   train=False)

    history["tr_loss"].append(tr_loss); history["tr_dice"].append(tr_dice); history["tr_iou"].append(tr_iou)
    history["va_loss"].append(va_loss); history["va_dice"].append(va_dice); history["va_iou"].append(va_iou)

    if va_dice > best_val_dice:
        best_val_dice = va_dice
        best_ckpt = os.path.join(OUTPUT, "unet_best.pt")
        torch.save({"model":model.state_dict(),"epoch":epoch,"val_dice":va_dice}, best_ckpt)

    print(f"[{epoch:03d}/{EPOCHS}] "
          f"train L{tr_loss:.4f} D{tr_dice:.4f} IoU{tr_iou:.4f} | "
          f"val L{va_loss:.4f} D{va_dice:.4f} IoU{va_iou:.4f} | "
          f"{time.time()-t0:.1f}s")

print("Best val Dice:", best_val_dice, "| ckpt:", best_ckpt)


In [ ]:
# best 가중치 로드
if best_ckpt and os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded epoch {ckpt['epoch']} | val_dice={ckpt['val_dice']:.4f}")

# 테스트 평가
model.eval()
total_loss=total_dice=total_iou=n=0
with torch.no_grad():
    for imgs, masks, _ in test_ld:
        imgs = imgs.to(device); masks = masks.to(device)
        logits = model(imgs)
        loss   = criterion(logits, masks)
        bs = imgs.size(0)
        total_loss += loss.item()*bs
        total_dice += dice_coeff(logits, masks)*bs
        total_iou  += iou_score(logits, masks)*bs
        n += bs
test_loss = total_loss/n; test_dice = total_dice/n; test_iou = total_iou/n
print(f"[TEST] Loss {test_loss:.4f} | Dice {test_dice:.4f} | IoU {test_iou:.4f}")

# 샘플 시각화
def viz_batch(imgs, masks, logits, n=6):
    n = min(n, imgs.size(0))
    probs = torch.sigmoid(logits).cpu()
    preds = (probs > 0.5).float()
    imgs_np  = imgs.cpu().permute(0,2,3,1).numpy()
    imgs_np  = imgs_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
    imgs_np  = np.clip(imgs_np,0,1)

    plt.figure(figsize=(12, 6))
    for i in range(n):
        # 원본
        ax = plt.subplot(3, n, i+1)
        ax.imshow(imgs_np[i]); ax.set_title("image"); ax.axis("off")
        # GT
        ax = plt.subplot(3, n, n+i+1)
        ax.imshow(masks[i,0].cpu(), cmap="gray"); ax.set_title("mask"); ax.axis("off")
        # Pred
        ax = plt.subplot(3, n, 2*n+i+1)
        ax.imshow(preds[i,0], cmap="gray"); ax.set_title("pred"); ax.axis("off")
    plt.tight_layout(); plt.show()

# 한 배치 시각화
imgs, masks, paths = next(iter(test_ld))
with torch.no_grad():
    logits = model(imgs.to(device))
viz_batch(imgs, masks, logits, n=6)


In [ ]:
# =========================
# BUSI 탐지(Detection) — 단일 셀 베이스라인
# - benign/malignant: *_mask 로부터 연결성분 → bbox 생성
# - normal: bbox 없음(배경), 거짓양성 억제에 도움
# - 모델: torchvision Faster R-CNN (ResNet-50 FPN, ImageNet 사전학습)
# - 입력 크기: 고정 리사이즈(IMG_SIZE), 이미지/마스크 동일 변환 → 좌표 일관성 유지
# - 평가: IoU@0.5에서 greedy matching으로 Precision/Recall/F1 산출 (라이트 버전)
# =========================

import os, random, time
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.ops import box_iou
from torchvision import models

# ------- 설정 -------
DATA_ROOT   = DATA_ROOT if 'DATA_ROOT' in globals() else "./data/BUSI/Dataset_BUSI_with_GT"  # benign/malignant/normal
OUTPUT      = "./outputs_busi_det"
IMG_SIZE    = 640          # 탐지엔 넉넉한 해상도 권장 (e.g., 512~768)
BATCH_SIZE  = 4            # detection은 메모리 큼: 2~6 범위 권장
EPOCHS      = 20
LR          = 5e-4
WEIGHT_DECAY= 1e-4
SPLIT_RATIO = (0.7, 0.1, 0.2)
SEED        = 42
AMP         = True
SCORE_THR   = 0.25         # 평가/시각화시 confidence threshold

os.makedirs(OUTPUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=False
    torch.backends.cudnn.benchmark=True
set_seed(SEED)

# ------- 1) 페어 만들기 (img, mask_or_None, has_lesion) -------
def list_pairs_with_normal(root):
    root = Path(root)
    pairs = []
    for cls in ["benign", "malignant"]:
        d = root/cls
        if not d.exists():
            print(f"[경고] 폴더 없음: {d}")
            continue
        mask_idx = {p.name.replace("_mask",""): str(p) for p in d.rglob("*_mask.*")}
        for p in d.rglob("*.*"):
            name = p.name
            if name.endswith((".png",".jpg",".jpeg",".bmp",".tif")) and "_mask" not in name:
                if name in mask_idx:
                    pairs.append((str(p), mask_idx[name], 1))
    nd = root/"normal"
    if nd.exists():
        for p in nd.rglob("*.*"):
            if p.name.endswith((".png",".jpg",".jpeg",".bmp",".tif")):
                pairs.append((str(p), None, 0))
    else:
        print(f"[주의] normal 폴더를 찾지 못했습니다: {nd}")
    return pairs

pairs_all = list_pairs_with_normal(DATA_ROOT)
print("총 샘플 수:", len(pairs_all),
      "| 양성:", sum(1 for _,m,h in pairs_all if h==1),
      "| 노말:", sum(1 for _,m,h in pairs_all if h==0))

# ------- 2) 분할 -------
def stratified_split_pairs(pairs, split=(0.7,0.1,0.2), seed=42):
    rng = random.Random(seed)
    pos = [it for it in pairs if it[2]==1]
    neg = [it for it in pairs if it[2]==0]
    rng.shuffle(pos); rng.shuffle(neg)
    def take(lst):
        n=len(lst); n_tr=int(n*split[0]); n_va=int(n*split[1])
        return lst[:n_tr], lst[n_tr:n_tr+n_va], lst[n_tr+n_va:]
    tr_p, va_p, te_p = take(pos)
    tr_n, va_n, te_n = take(neg)
    train, val, test = tr_p+tr_n, va_p+va_n, te_p+te_n
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

train_items, val_items, test_items = stratified_split_pairs(pairs_all, split=SPLIT_RATIO, seed=SEED)
print("train/val/test:", len(train_items), len(val_items), len(test_items))

# ------- 3) 마스크 → bbox 유틸 -------
def mask_to_boxes(mask_np, min_area=30):
    """
    mask_np: 2D numpy (0/255 or 0/1)
    연결성분을 찾아 외접 bbox(xmin,ymin,xmax,ymax) 리스트 반환.
    너무 작은 노이즈는 min_area로 필터링.
    """
    m = (mask_np > 0).astype(np.uint8)
    if m.max() == 0:
        return []
    # 연결성분 라벨링 (간단 구현)
    # 여기서는 scipy 없이 수제 구현 대신, contours 기반으로도 가능하지만
    # 간단히 np.where를 사용해 bounding box를 뽑자(물론 성능은 준수)
    # 전략: find all connected components via BFS (작은 데이터라 충분)
    H, W = m.shape
    visited = np.zeros_like(m, dtype=np.uint8)
    boxes = []
    from collections import deque
    for y in range(H):
        for x in range(W):
            if m[y,x]==1 and not visited[y,x]:
                q=deque([(y,x)]); visited[y,x]=1
                ymin=y; ymax=y; xmin=x; xmax=x; area=0
                while q:
                    cy,cx=q.popleft()
                    area += 1
                    if cy<ymin: ymin=cy
                    if cy>ymax: ymax=cy
                    if cx<xmin: xmin=cx
                    if cx>xmax: xmax=cx
                    for ny,nx in [(cy-1,cx),(cy+1,cx),(cy,cx-1),(cy,cx+1)]:
                        if 0<=ny<H and 0<=nx<W and m[ny,nx]==1 and not visited[ny,nx]:
                            visited[ny,nx]=1
                            q.append((ny,nx))
                if area >= min_area:
                    boxes.append([xmin, ymin, xmax+1, ymax+1])  # +1 to make max exclusive-style
    return boxes

# ------- 4) Dataset (고정 리사이즈 후 bbox 생성) -------
class BUSIDetDataset(Dataset):
    """
    - 이미지를 IMG_SIZE로 리사이즈한 후, 마스크도 동일 리사이즈 → 그 위에서 bbox 산출
    - 출력 형식은 torchvision detection 포맷:
      images: Tensor [3,H,W] (0~1)
      targets: dict(boxes [N,4], labels [N], image_id, area [N], iscrowd [N])
      레이블은 단일 클래스 "lesion"=1
    """
    def __init__(self, items, img_size=640, is_train=False):
        self.items = items
        self.size = img_size
        self.is_train = is_train
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        img_path, mask_path, has_lesion = self.items[idx]

        img = Image.open(img_path).convert("RGB")
        img = TF.resize(img, (self.size, self.size), interpolation=TF.InterpolationMode.BILINEAR)

        if mask_path is None:
            mask = Image.fromarray(np.zeros((self.size,self.size), dtype=np.uint8))
        else:
            m = Image.open(mask_path).convert("L")
            mask = TF.resize(m, (self.size, self.size), interpolation=TF.InterpolationMode.NEAREST)

        # 약한 증강(수평플립/회전)
        if self.is_train:
            if random.random()<0.5:
                img  = TF.hflip(img); mask = TF.hflip(mask)
            ang = random.uniform(-10,10)
            img  = TF.rotate(img, ang, interpolation=TF.InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, ang, interpolation=TF.InterpolationMode.NEAREST)

        # to tensor
        img_t = TF.to_tensor(img)  # [0,1]
        mask_np = np.array(mask, dtype=np.uint8)

        # mask -> boxes
        boxes = mask_to_boxes(mask_np, min_area=30) if has_lesion==1 else []
        if len(boxes)==0:
            boxes_t = torch.zeros((0,4), dtype=torch.float32)
            labels_t= torch.zeros((0,), dtype=torch.int64)
            area_t  = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t= torch.ones((len(boxes),), dtype=torch.int64)   # class=1
            # area = (xmax - xmin) * (ymax - ymin)
            wh = (boxes_t[:,2]-boxes_t[:,0]) * (boxes_t[:,3]-boxes_t[:,1])
            area_t  = wh.to(torch.float32)
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([idx]),
            "area": area_t,
            "iscrowd": iscrowd
        }
        return img_t, target, img_path

def collate_fn(batch):
    imgs, tars, paths = list(zip(*batch))
    return list(imgs), list(tars), list(paths)

train_ds = BUSIDetDataset(train_items, IMG_SIZE, is_train=True)
val_ds   = BUSIDetDataset(val_items,   IMG_SIZE, is_train=False)
test_ds  = BUSIDetDataset(test_items,  IMG_SIZE, is_train=False)

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, collate_fn=collate_fn)
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, collate_fn=collate_fn)
test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, collate_fn=collate_fn)

# ------- 5) 모델 -------
def build_model(num_classes=2):
    """
    num_classes=2 (background + lesion)
    torchvision Faster R-CNN은 background를 내부적으로 처리하므로
    여기서는 레이블 1만 사용하지만 모델 헤드는 2-class로 만들어둔다.
    """
    model = models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # 새 predictor로 교체
    model.roi_heads.box_predictor = models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    return model

model = build_model(num_classes=2).to(device)
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=AMP)

# ------- 6) 학습/검증 루프 -------
def train_one_epoch(model, loader):
    model.train()
    loss_hist=[]
    for imgs, targets, _ in loader:
        imgs = [im.to(device) for im in imgs]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

        with torch.cuda.amp.autocast(enabled=AMP):
            loss_dict = model(imgs, targets)   # dict of losses
            loss = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        loss_hist.append(loss.item())

    return float(np.mean(loss_hist))

@torch.no_grad()
def simple_eval(model, loader, score_thr=0.25, iou_thr=0.5):
    """
    간단 평가: IoU@0.5에서 greedy matching으로 TP/FP/FN 산출 → P/R/F1
    - 단일 클래스 가정
    """
    model.eval()
    total_TP=total_FP=total_FN=0
    for imgs, targets, _ in loader:
        imgs = [im.to(device) for im in imgs]
        outs = model(imgs)

        for out, tgt in zip(outs, targets):
            # GT
            gtb = tgt["boxes"].to(device)
            # Pred (score filtering)
            keep = out["scores"] >= score_thr
            pb = out["boxes"][keep].to(device)

            if len(gtb)==0 and len(pb)==0:
                continue
            if len(gtb)==0 and len(pb)>0:
                total_FP += len(pb); continue
            if len(gtb)>0 and len(pb)==0:
                total_FN += len(gtb); continue

            ious = box_iou(pb, gtb)  # [P, G]
            # greedy matching
            matched_gt = set()
            matched_pr = set()
            for pi in range(ious.size(0)):
                gi = torch.argmax(ious[pi]).item()
                if ious[pi,gi].item() >= iou_thr and gi not in matched_gt:
                    matched_pr.add(pi); matched_gt.add(gi)
            TP = len(matched_pr)
            FP = len(pb) - TP
            FN = len(gtb) - TP
            total_TP += TP; total_FP += FP; total_FN += FN

    prec = total_TP / (total_TP + total_FP + 1e-9)
    rec  = total_TP / (total_TP + total_FN + 1e-9)
    f1   = 2*prec*rec / (prec+rec+1e-9)
    return {"P":prec, "R":rec, "F1":f1, "TP":total_TP, "FP":total_FP, "FN":total_FN}

best_f1, best_path = -1, None
for epoch in range(1, EPOCHS+1):
    t0=time.time()
    tr_loss = train_one_epoch(model, train_ld)
    val_metrics = simple_eval(model, val_ld, score_thr=SCORE_THR, iou_thr=0.5)

    if val_metrics["F1"] > best_f1:
        best_f1 = val_metrics["F1"]
        best_path = os.path.join(OUTPUT, "fasterrcnn_busi_best.pt")
        torch.save({"model": model.state_dict(),
                    "epoch": epoch,
                    "val_f1": best_f1}, best_path)

    print(f"[{epoch:02d}/{EPOCHS}] "
          f"train loss {tr_loss:.4f} | "
          f"val P {val_metrics['P']:.3f} R {val_metrics['R']:.3f} F1 {val_metrics['F1']:.3f} "
          f"| {time.time()-t0:.1f}s")

print("Best val F1:", best_f1, "| ckpt:", best_path)

# ------- 7) 테스트 평가 -------
if best_path and os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded best (epoch {ckpt['epoch']}, val_f1={ckpt['val_f1']:.3f})")

test_metrics = simple_eval(model, test_ld, score_thr=SCORE_THR, iou_thr=0.5)
print(f"[TEST @IoU0.5] P {test_metrics['P']:.3f} | R {test_metrics['R']:.3f} | F1 {test_metrics['F1']:.3f} "
      f"| TP {test_metrics['TP']} FP {test_metrics['FP']} FN {test_metrics['FN']}")

# ------- 8) 시각화 (GT vs Pred) -------
@torch.no_grad()
def show_detections(dataset, n=6, score_thr=0.25):
    model.eval()
    idxs = np.random.choice(len(dataset), size=min(n, len(dataset)), replace=False)
    plt.figure(figsize=(14, 2.5*len(idxs)))
    row = 0
    for i, idx in enumerate(idxs):
        img, tgt, path = dataset[idx]
        out = model([img.to(device)])[0]
        keep = out["scores"].cpu().numpy() >= score_thr
        boxes = out["boxes"].cpu().numpy()[keep]
        scores= out["scores"].cpu().numpy()[keep]
        gtb   = tgt["boxes"].numpy()

        # denorm (이미 0~1이므로 바로 numpy로)
        im = img.permute(1,2,0).numpy()
        im = np.clip(im, 0, 1)

        # 그리기
        ax = plt.subplot(len(idxs), 1, i+1)
        ax.imshow(im); ax.axis("off")
        # GT(녹색), Pred(빨강)
        for (xmin,ymin,xmax,ymax) in gtb:
            ax.add_patch(plt.Rectangle((xmin,ymin), xmax-xmin, ymax-ymin,
                                       fill=False, linewidth=2))
        for (xmin,ymin,xmax,ymax), sc in zip(boxes, scores):
            rect = plt.Rectangle((xmin,ymin), xmax-xmin, ymax-ymin,
                                 fill=False, linewidth=2, linestyle="--")
            rect.set_edgecolor("red"); ax.add_patch(rect)
            ax.text(xmin, ymin-3, f"{sc:.2f}", color="red", fontsize=8,
                    bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
        ax.set_title(f"{Path(path).name}  |  GT:{len(gtb)}  Pred(>{score_thr}):{len(boxes)}")
    plt.tight_layout(); plt.show()

show_detections(test_ds, n=6, score_thr=SCORE_THR)


In [ ]:
# =========================
# BUSI 분할 (노말 포함) — 통합 실행 셀 (수정 버전)
# 핵심 수정:
#  1) Up 블록 인터페이스를 Up(ch_in, ch_skip, out_ch)로 변경해 concat 후 채널수 정확 반영
#  2) 최종 출력은 표준 U-Net 스타일(u4 → 1x1 conv), 마지막에 x1을 재-concat 하는 트릭 제거
#  3) WeightedRandomSampler로 양성 비율을 높이고, BCE pos_weight + Dice로 불균형 완화
# =========================

import os, random, time
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF

import matplotlib.pyplot as plt

# ===== 하이퍼파라미터 / 경로 =====
DATA_ROOT   = DATA_ROOT if 'DATA_ROOT' in globals() else "./data/BUSI/Dataset_BUSI_with_GT"
OUTPUT      = "./outputs_busi_seg_normal_fixed"
IMG_SIZE    = 384
BATCH_SIZE  = 8
EPOCHS      = 60
LR          = 3e-4
WEIGHT_DECAY= 1e-4
SPLIT_RATIO = (0.7, 0.1, 0.2)   # train / val / test
SEED        = 42
AMP         = True
POS_WEIGHT  = 3.0               # BCE 양성가중(2~5 탐색 권장)
POS_RATIO_TRAIN = 0.5           # 훈련 배치에서 양성(lesion) 목표 비율(샘플러 가중으로 근사)

os.makedirs(OUTPUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
set_seed(SEED)

# ===== 1) 데이터 페어 구성 =====
# 반환: (img_path, mask_path_or_None, has_lesion)
#  - has_lesion: 1(benign/malignant), 0(normal)
def list_pairs_with_normal(root):
    root = Path(root)
    pairs = []
    for cls in ["benign", "malignant"]:
        d = root/cls
        if not d.exists():
            print(f"[경고] 폴더 없음: {d}")
            continue
        mask_idx = {p.name.replace("_mask",""): str(p) for p in d.rglob("*_mask.*")}
        for p in d.rglob("*.*"):
            name = p.name
            if name.endswith((".png",".jpg",".jpeg",".bmp",".tif")) and "_mask" not in name:
                if name in mask_idx:
                    pairs.append((str(p), mask_idx[name], 1))
    nd = root/"normal"
    if nd.exists():
        for p in nd.rglob("*.*"):
            name = p.name
            if name.endswith((".png",".jpg",".jpeg",".bmp",".tif")):
                pairs.append((str(p), None, 0))
    else:
        print(f"[주의] normal 폴더를 찾지 못했습니다: {nd}")
    return pairs

pairs_all = list_pairs_with_normal(DATA_ROOT)
print("총 샘플 수:", len(pairs_all),
      "| 양성:", sum(1 for _,m,h in pairs_all if h==1),
      "| 노말:", sum(1 for _,m,h in pairs_all if h==0))

# ===== 2) 라벨별 계층 분할 =====
def stratified_split_pairs(pairs, split=(0.7,0.1,0.2), seed=42):
    rng = random.Random(seed)
    pos = [it for it in pairs if it[2]==1]
    neg = [it for it in pairs if it[2]==0]
    rng.shuffle(pos); rng.shuffle(neg)
    def take(lst):
        n=len(lst); n_tr=int(n*split[0]); n_va=int(n*split[1])
        return lst[:n_tr], lst[n_tr:n_tr+n_va], lst[n_tr+n_va:]
    tr_p, va_p, te_p = take(pos)
    tr_n, va_n, te_n = take(neg)
    train, val, test = tr_p+tr_n, va_p+va_n, te_p+te_n
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

train_items, val_items, test_items = stratified_split_pairs(pairs_all, split=SPLIT_RATIO, seed=SEED)
print("train/val/test:", len(train_items), len(val_items), len(test_items))

# ===== 3) Dataset (노말은 0마스크 생성, 이미지/마스크 동일 증강) =====
class BUSISegDataset(Dataset):
    def __init__(self, items, img_size=384, is_train=False):
        self.items = items
        self.size = img_size
        self.is_train = is_train
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        img_path, mask_path, has_lesion = self.items[idx]
        img = Image.open(img_path).convert("RGB")
        img = TF.resize(img, (self.size, self.size), interpolation=TF.InterpolationMode.BILINEAR)
        if mask_path is None:
            mask = Image.fromarray(np.zeros((self.size, self.size), dtype=np.uint8))
        else:
            mask = Image.open(mask_path).convert("L")
            mask = TF.resize(mask, (self.size, self.size), interpolation=TF.InterpolationMode.NEAREST)
        if self.is_train:
            if random.random() < 0.5:
                img = TF.hflip(img); mask = TF.hflip(mask)
            ang = random.uniform(-10, 10)
            img  = TF.rotate(img, ang, interpolation=TF.InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, ang, interpolation=TF.InterpolationMode.NEAREST)
            if random.random() < 0.5:
                img = TF.adjust_brightness(img, 1.0 + random.uniform(-0.1,0.1))
                img = TF.adjust_contrast(img,  1.0 + random.uniform(-0.1,0.1))
        img  = TF.to_tensor(img)
        img  = TF.normalize(img, [0.485,0.456,0.406],[0.229,0.224,0.225])
        mask = (TF.to_tensor(mask) > 0.5).float()  # [1,H,W] 이진
        return img, mask, has_lesion, img_path

train_ds = BUSISegDataset(train_items, IMG_SIZE, is_train=True)
val_ds   = BUSISegDataset(val_items,   IMG_SIZE, is_train=False)
test_ds  = BUSISegDataset(test_items,  IMG_SIZE, is_train=False)

# ===== 4) 샘플러(훈련): 양성 비율을 올려 '배경 예측' 붕괴 방지 =====
# train_items: (img_path, mask_path or None, has_lesion)  ← 3-튜플임!
# 양성(1)에는 POS_RATIO_TRAIN, 노말(0)엔 1-POS_RATIO_TRAIN 가중
import torch
from torch.utils.data import WeightedRandomSampler

if any(len(t) != 3 for t in train_items):
    raise ValueError(f"train_items는 3-튜플이어야 합니다. 예: (img_path, mask_path, has_lesion). "
                     f"예시 첫 항목: {train_items[0]}")

weights_list = [
    (POS_RATIO_TRAIN if has_lesion == 1 else (1.0 - POS_RATIO_TRAIN))
    for _, _, has_lesion in train_items
]

# (권장) double 텐서로 변환
weights = torch.tensor(weights_list, dtype=torch.double)
sampler = WeightedRandomSampler(weights=weights,
                                num_samples=len(train_items),  # 한 에폭당 샘플 수
                                replacement=True)

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                      num_workers=4, pin_memory=True)

# 검증/테스트는 셔플 없음
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True)
test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True)

# ===== 5) U-Net (same padding) — 채널 수 정확히 처리한 버전 =====
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=3):
        super().__init__()
        p = k//2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x): return self.conv(self.pool(x))

class Up(nn.Module):
    """업샘플 → skip concat → DoubleConv
       ch_in: 업샘플된 텐서 채널, ch_skip: 스킵 채널, out_ch: 출력 채널"""
    def __init__(self, ch_in, ch_skip, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = DoubleConv(ch_in + ch_skip, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        if dy!=0 or dx!=0:
            x = TF.pad(x, [0,0,max(dx,0),max(dy,0)])
            x = x[:, :, :skip.size(2), :skip.size(3)]
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, base=64, num_classes=1):
        super().__init__()
        self.inc   = DoubleConv(in_ch, base)         # b
        self.down1 = Down(base, base*2)              # 2b
        self.down2 = Down(base*2, base*4)            # 4b
        self.down3 = Down(base*4, base*8)            # 8b
        self.down4 = Down(base*8, base*16)           # 16b

        self.up1 = Up(base*16, base*8,  base*8)      # (16b up)+8b -> 8b
        self.up2 = Up(base*8,  base*4,  base*4)      # (8b  up)+4b -> 4b
        self.up3 = Up(base*4,  base*2,  base*2)      # (4b  up)+2b -> 2b
        self.up4 = Up(base*2,  base,    base)        # (2b  up)+ b ->  b

        self.outc = nn.Conv2d(base, num_classes, 1)  # 표준 U-Net 출력(head)

    def forward(self, x):
        x1 = self.inc(x)    # b
        x2 = self.down1(x1) # 2b
        x3 = self.down2(x2) # 4b
        x4 = self.down3(x3) # 8b
        x5 = self.down4(x4) # 16b
        u1 = self.up1(x5, x4)
        u2 = self.up2(u1, x3)
        u3 = self.up3(u2, x2)
        u4 = self.up4(u3, x1)
        return self.outc(u4)

# (선택) shape sanity check
with torch.no_grad():
    _x = torch.randn(2,3,IMG_SIZE,IMG_SIZE).to(device)
    _m = UNet().to(device)
    _y = _m(_x)
    print("sanity out shape:", _y.shape)

# ===== 6) 손실/지표 =====
class BCEDiceLoss(nn.Module):
    def __init__(self, dice_weight=0.5, pos_weight=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
        self.dw = dice_weight
    def forward(self, logits, target):
        bce = self.bce(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(1,2,3))
        union = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
        dice = 1 - (2*inter + 1e-6)/(union + 1e-6)
        return (1-self.dw)*bce + self.dw*dice.mean()

def dice_coeff_bin(logits, target, thr=0.5, eps=1e-6):
    prob = torch.sigmoid(logits); pred = (prob > thr).float()
    inter = (pred*target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return ((2*inter + eps)/(union + eps))

def iou_bin(logits, target, thr=0.5, eps=1e-6):
    prob = torch.sigmoid(logits); pred = (prob > thr).float()
    inter = (pred*target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    return ((inter + eps)/(union + eps))

def fp_rate_on_normal(logits, target, thr=0.5):
    # normal(=target.sum()==0)에서 예측 양성 픽셀 비율
    prob = torch.sigmoid(logits); pred = (prob > thr).float()
    H = pred.size(2)*pred.size(3)
    is_normal = (target.sum(dim=(1,2,3)) == 0)
    if is_normal.any():
        fp_pixel = pred[is_normal].sum(dim=(1,2,3))
        return (fp_pixel / H).mean().item()
    return None

# ===== 7) 학습/평가 루프 =====
model = UNet(in_ch=3, base=64, num_classes=1).to(device)
criterion = BCEDiceLoss(dice_weight=0.5, pos_weight=POS_WEIGHT)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=AMP)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss=0.0; pos_dice=[]; pos_iou=[]; normal_fp=[]; n=0
    for imgs, masks, has_lesion, _ in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks= masks.to(device, non_blocking=True)
        has_lesion = has_lesion.to(device)

        with torch.cuda.amp.autocast(enabled=AMP):
            logits = model(imgs)
            loss   = criterion(logits, masks)

        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()

        # 양성 케이스 Dice/IoU
        pos_mask = (has_lesion==1)
        if pos_mask.any():
            pos_dice.append(dice_coeff_bin(logits[pos_mask], masks[pos_mask]).mean().item())
            pos_iou.append(iou_bin(logits[pos_mask],   masks[pos_mask]).mean().item())

        # 노말 케이스 FP-rate
        fp = fp_rate_on_normal(logits, masks)
        if fp is not None: normal_fp.append(fp)

        bs = imgs.size(0); total_loss += loss.item()*bs; n += bs

    avg_loss = total_loss/n
    avg_dice = float(np.mean(pos_dice)) if len(pos_dice)>0 else float('nan')
    avg_iou  = float(np.mean(pos_iou))  if len(pos_iou)>0 else float('nan')
    avg_fp   = float(np.mean(normal_fp)) if len(normal_fp)>0 else float('nan')
    return avg_loss, avg_dice, avg_iou, avg_fp

best_val_dice, best_ckpt = -1, None
history = {"tr_loss":[], "tr_dice":[], "tr_iou":[], "tr_fp":[],
           "va_loss":[], "va_dice":[], "va_iou":[], "va_fp":[]}

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss,tr_dice,tr_iou,tr_fp = run_epoch(train_ld, train=True)
    va_loss,va_dice,va_iou,va_fp = run_epoch(val_ld,   train=False)

    history["tr_loss"].append(tr_loss); history["tr_dice"].append(tr_dice)
    history["tr_iou"].append(tr_iou);  history["tr_fp"].append(tr_fp)
    history["va_loss"].append(va_loss); history["va_dice"].append(va_dice)
    history["va_iou"].append(va_iou);  history["va_fp"].append(va_fp)

    if not np.isnan(va_dice) and va_dice > best_val_dice:
        best_val_dice = va_dice
        best_ckpt = os.path.join(OUTPUT, "unet_best_normal.pt")
        torch.save({"model":model.state_dict(),"epoch":epoch,"val_dice":va_dice}, best_ckpt)

    print(f"[{epoch:03d}/{EPOCHS}] "
          f"train L{tr_loss:.4f} D{tr_dice:.4f} IoU{tr_iou:.4f} FP{tr_fp if not np.isnan(tr_fp) else float('nan'):.4f} | "
          f"val   L{va_loss:.4f} D{va_dice:.4f} IoU{va_iou:.4f} FP{va_fp if not np.isnan(va_fp) else float('nan'):.4f} | "
          f"{time.time()-t0:.1f}s")

print("Best val Dice(양성):", best_val_dice, "| ckpt:", best_ckpt)

# ===== 8) 테스트 평가 & 시각화 =====
if best_ckpt and os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded epoch {ckpt['epoch']} | val_dice={ckpt['val_dice']:.4f}")

model.eval()
test_loss,test_dice,test_iou,test_fp = run_epoch(test_ld, train=False)
print(f"[TEST] Loss {test_loss:.4f} | Dice_pos {test_dice:.4f} | IoU_pos {test_iou:.4f} | FP_rate_normal {test_fp:.6f}")

def viz_batch(imgs, masks, logits, has_lesion, n=6):
    n = min(n, imgs.size(0))
    probs = torch.sigmoid(logits).cpu()
    preds = (probs > 0.5).float()
    imgs_np = imgs.cpu().permute(0,2,3,1).numpy()
    imgs_np = imgs_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
    imgs_np = np.clip(imgs_np, 0, 1)
    plt.figure(figsize=(12, 6))
    for i in range(n):
        ax = plt.subplot(3, n, i+1); ax.imshow(imgs_np[i])
        ax.set_title(f"image | {'pos' if has_lesion[i]==1 else 'normal'}"); ax.axis("off")
        ax = plt.subplot(3, n, n+i+1); ax.imshow(masks[i,0].cpu(), cmap="gray"); ax.set_title("mask"); ax.axis("off")
        ax = plt.subplot(3, n, 2*n+i+1); ax.imshow(preds[i,0], cmap="gray"); ax.set_title("pred"); ax.axis("off")
    plt.tight_layout(); plt.show()

with torch.no_grad():
    imgs, masks, has_lesion, paths = next(iter(test_ld))
    logits = model(imgs.to(device))
viz_batch(imgs, masks, logits, has_lesion, n=6)
